# Step 2 Quantized arithmetic

Open this notebook in Google Colab and choose **Runtime → Run all**. Each code
cell shows its results directly below it. The final cell runs 13 checks and
should finish with **OK** and **All 13 arithmetic checks passed.**

Everything needed is inside this notebook. No repository clone, package install,
dataset, GPU, or Drive mount is needed. The arithmetic functions and checks match
the project's Python files.

The examples explain the calculations that our accelerator will later perform.
They do not measure accelerator speed or digit-recognition accuracy.


## 1 Small inputs and a wider running total

**INT8** represents whole numbers from −128 to 127. **INT32** represents a wider
range, from −2,147,483,648 to 2,147,483,647. We use INT8 inputs and an INT32 running
total, called an **accumulator**.

The cell below defines those rules. Python integers are not physically packed
into eight bits here; the checks enforce the modeled ranges.


In [1]:
from dataclasses import dataclass
import math
import struct

INT8_MIN, INT8_MAX = -128, 127
INT32_MIN, INT32_MAX = -(2**31), 2**31 - 1


def validate_int8(value):
    if type(value) is not int:
        raise TypeError("INT8 operands must be integers, not floats or booleans.")
    if not INT8_MIN <= value <= INT8_MAX:
        raise ValueError(f"Operand {value} is outside the signed INT8 range.")


def checked_mac(accumulator, left, right):
    """One multiply-and-add; flag overflow rather than use Python's wider integers."""
    validate_int8(left)
    validate_int8(right)
    if type(accumulator) is not int:
        raise TypeError("The accumulator must be an integer.")
    if not INT32_MIN <= accumulator <= INT32_MAX:
        raise OverflowError("Initial accumulator is outside the INT32 range.")
    result = accumulator + left * right
    if not INT32_MIN <= result <= INT32_MAX:
        raise OverflowError("INT32 accumulation overflow.")
    return result

print(f"INT8 inputs: {INT8_MIN} to {INT8_MAX}")
print(f"INT32 total: {INT32_MIN} to {INT32_MAX}")


INT8 inputs: -128 to 127
INT32 total: -2147483648 to 2147483647


## 2 Follow one calculation

A **multiply-accumulate**, or MAC, multiplies two inputs and adds the product to
the running total. A **dot product** repeats that operation for pairs of inputs.

This example needs three MACs:

`2 × 4 + 3 × 5 + 1 × 6 = 8 + 15 + 6 = 29`

The output shows the running total changing from **8** to **23** to **29**.


In [2]:
def int8_dot(left, right):
    if not left or len(left) != len(right):
        raise ValueError("Dot-product vectors must have equal, nonzero lengths.")
    accumulator = 0
    for a, b in zip(left, right):
        accumulator = checked_mac(accumulator, a, b)
    return accumulator

inputs = [2, 3, 1]
weights = [4, 5, 6]
total = 0
for left, right in zip(inputs, weights):
    total = checked_mac(total, left, right)
    print(f"{left} * {right} = {left * right}; running total = {total}")
print(f"Dot-product answer: {int8_dot(inputs, weights)}")


2 * 4 = 8; running total = 8
3 * 5 = 15; running total = 23
1 * 6 = 6; running total = 29
Dot-product answer: 29


## 3 Why the accumulator is wider

Even valid INT8 inputs can produce a larger result: **127 × 127 = 16,129**.
That does not fit in INT8, but it fits in INT32.

The following cell also deliberately tries an invalid input and an overflowing
total. Those expected errors are caught and printed so **Run all** can continue.


In [3]:
print(f"127 * 127 = {checked_mac(0, 127, 127)} (fits INT32)")
try:
    checked_mac(0, 128, 1)
except ValueError as error:
    print(f"Expected input error: {error}")
try:
    checked_mac(INT32_MAX, 1, 1)
except OverflowError as error:
    print(f"Expected overflow error: {error}")


127 * 127 = 16129 (fits INT32)
Expected input error: Operand 128 is outside the signed INT8 range.
Expected overflow error: INT32 accumulation overflow.


## 4 Repeat the calculation for a matrix

A **matrix** is a rectangular table of numbers. Matrix multiplication calculates
several dot products. The first output below is the same **29** as before.

Expected result: `[[29, 6], [8, 4]]`.


In [4]:
def matrix_shape(matrix):
    """Reject empty or ragged matrices rather than silently truncating data."""
    if not matrix or not matrix[0]:
        raise ValueError("A matrix must have at least one row and column.")
    columns = len(matrix[0])
    if any(len(row) != columns for row in matrix):
        raise ValueError("All matrix rows must have the same length.")
    return len(matrix), columns


def multiplication_shape(left, right):
    rows, inner = matrix_shape(left)
    right_rows, columns = matrix_shape(right)
    if inner != right_rows:
        raise ValueError("Left columns must equal right rows.")
    return rows, inner, columns


def int8_matmul(left, right):
    rows, inner, columns = multiplication_shape(left, right)
    return [
        [int8_dot(left[row], [right[k][column] for k in range(inner)])
         for column in range(columns)]
        for row in range(rows)
    ]

left = [[2, 3, 1], [-1, 0, 2]]
right = [[4, -2], [5, 3], [6, 1]]
for row in int8_matmul(left, right):
    print(row)


[29, 6]
[8, 4]


## 5 Represent fractions using an integer and a scale

**Quantization** stores an integer together with a scale. A stored integer of 12
with scale 0.1 represents `12 × 0.1 = 1.2`.

The quantizer divides by the scale, clamps to the allowed range, and rounds.
Our symmetric quantizer uses −127 through 127 and reserves −128. The arithmetic
functions above still accept the full INT8 range.

This cell defines that rule; the following cells show its effects.


In [5]:
@dataclass(frozen=True)
class QuantizedMatrix:
    values: list[list[int]]
    scale: float
    clipped_values: int


def quantize_int8(matrix, scale=None):
    """One scale per matrix, zero point 0, ties-to-even rounding, range [-127,127].

    The symmetric quantizer reserves -128. The arithmetic units still support
    the entire signed INT8 range. Explicit scales allow later calibration.
    """
    matrix_shape(matrix)
    real_values = [[float(value) for value in row] for row in matrix]
    if any(not math.isfinite(value) for row in real_values for value in row):
        raise ValueError("Quantization inputs must be finite.")
    maximum = max(abs(value) for row in real_values for value in row)
    if scale is None:
        scale = maximum / INT8_MAX if maximum else 1.0
    scale = float(scale)
    if not math.isfinite(scale) or scale <= 0:
        raise ValueError("Quantization scale must be finite and positive.")
    clipped = 0
    values = []
    for row in real_values:
        quantized_row = []
        for value in row:
            scaled = value / scale
            # Clamp before round, which also avoids round(inf) for extreme ratios.
            clipped += int(scaled < -127 or scaled > 127)
            quantized_row.append(round(max(-127, min(127, scaled))))
        values.append(quantized_row)
    return QuantizedMatrix(values, scale, clipped)

print("Quantizer ready: integer range -127 to 127, one scale per matrix.")


Quantizer ready: integer range -127 to 127, one scale per matrix.


### Rounding loses some detail

With scale 0.1, **1.23** becomes stored integer **12**, which reconstructs to
**1.20**. The rounding error is **0.03**.

Exactly halfway cases round to the nearest even integer: **0.5 → 0** and
**1.5 → 2**.


In [6]:
original = 1.23
rounded = quantize_int8([[original]], scale=0.1)
stored = rounded.values[0][0]
reconstructed = stored * rounded.scale
print(f"Original: {original}")
print(f"Stored integer: {stored}; scale: {rounded.scale}")
print(f"Reconstructed: {reconstructed:.2f}; error: {abs(original - reconstructed):.2f}")
print("Halfway cases:", quantize_int8([[0.5, 1.5]], scale=1).values[0])


Original: 1.23
Stored integer: 12; scale: 0.1
Reconstructed: 1.20; error: 0.03
Halfway cases: [0, 2]


### Clipping and automatic scales

**Clipping** replaces a value outside the allowed range with the nearest endpoint.
With scale 1, −200 and 200 clip to −127 and 127.

When no scale is supplied, the code uses the largest absolute input divided by
127. An all-zero matrix uses scale 1 to avoid dividing by zero.


In [7]:
clipped = quantize_int8([[-200, 0, 200]], scale=1)
print("Clipped values:", clipped.values[0])
print("Number clipped:", clipped.clipped_values)
automatic = quantize_int8([[-2, 0, 2]])
print("Automatic scale:", automatic.scale)
print("Stored values:", automatic.values[0])
zeros = quantize_int8([[0, 0]])
print("All-zero input:", zeros.values[0], "with scale", zeros.scale)


Clipped values: [-127, 0, 127]
Number clipped: 2
Automatic scale: 0.015748031496062992
Stored values: [-127, 0, 127]
All-zero input: [0, 0] with scale 1.0


## 6 Both scales matter when multiplying

The inputs `[1.0, 1.5]` with scale 0.5 become `[2, 3]`. The weights `[1.0, 1.25]`
with scale 0.25 become `[4, 5]`.

The integer calculation gives `2 × 4 + 3 × 5 = 23`. To reconstruct the real
result, multiply by **both** scales: `23 × 0.5 × 0.25 = 2.875`.

These particular values fit their scales exactly. Other inputs can lose detail
through rounding or clipping.


In [8]:
def dequantize_accumulators(matrix, left_scale, right_scale):
    """Reconstruct real outputs; no bias, activation, or output requantization yet."""
    return [[value * left_scale * right_scale for value in row] for row in matrix]

left = quantize_int8([[1.0, 1.5]], scale=0.5)
right = quantize_int8([[1.0], [1.25]], scale=0.25)
integer_result = int8_matmul(left.values, right.values)
real_result = dequantize_accumulators(integer_result, left.scale, right.scale)
print("Integer result:", integer_result[0][0])
print("Reconstructed result:", real_result[0][0])
print("Direct calculation:", 1.0 * 1.0 + 1.5 * 1.25)


Integer result: 23
Reconstructed result: 2.875
Direct calculation: 2.875


## 7 Keep FP32 separate from ordinary Python floats

The later baseline uses **FP32**, a 32-bit floating-point representation.
Ordinary Python floats usually have more precision, so they are not an FP32
baseline. The helpers below explicitly round each multiply and addition to FP32.

For this example, the wider Python-float calculation retains a result of **1**;
the explicitly rounded FP32 calculation loses the small addition and returns **0**.


In [9]:
def fp32(value):
    """Round to IEEE binary32 using Python's standard library."""
    return struct.unpack("f", struct.pack("f", value))[0]


def fp32_mac(accumulator, left, right):
    """Separate rounded FP32 multiply and add; fused multiply-add is not modeled."""
    return fp32(accumulator + fp32(left * right))


def float_matmul(left, right):
    """Python float reference for the teaching demo; this is NOT an FP32 baseline."""
    rows, inner, columns = multiplication_shape(left, right)
    return [
        [sum(float(left[row][k]) * float(right[k][column]) for k in range(inner))
         for column in range(columns)]
        for row in range(rows)
    ]

values = [16777216.0, 1.0, -16777216.0]
fp32_total = 0.0
for value in values:
    fp32_total = fp32_mac(fp32_total, value, 1.0)
print("Explicit FP32 result:", fp32_total)
print("Python-float teaching reference:", float_matmul([values], [[1], [1], [1]])[0][0])


Explicit FP32 result: 0.0
Python-float teaching reference: 1.0


## 8 Run all arithmetic checks

These are the same 13 tests as the project's `tests/test_arithmetic.py`.
They check known answers, negative inputs, shapes, integer limits, rounding,
clipping, scales, and FP32 rounding. No other project files are loaded.

Expected final output: **OK**, then **All 13 arithmetic checks passed.**


In [10]:
import sys
import unittest


class ArithmeticTests(unittest.TestCase):
    def test_hand_calculated_dot_product(self):
        self.assertEqual(int8_dot([2, 3, 1], [4, 5, 6]), 29)

    def test_signed_endpoints_and_cancellation(self):
        self.assertEqual(int8_dot([-128, 127], [-128, -127]), 255)
        self.assertEqual(int8_dot([127, 127], [127, -127]), 0)

    def test_hand_calculated_matrix(self):
        self.assertEqual(int8_matmul([[2, 3, 1], [-1, 0, 2]],
                                    [[4, -2], [5, 3], [6, 1]]), [[29, 6], [8, 4]])

    def test_reject_invalid_shapes(self):
        for left, right in (([], [[1]]), ([[1, 2], [3]], [[1], [2]]),
                            ([[1, 2]], [[1]]), ([[]], [[1]])):
            with self.subTest(left=left, right=right), self.assertRaises(ValueError):
                int8_matmul(left, right)
        for left, right in (([], []), ([1, 2], [1])):
            with self.assertRaises(ValueError):
                int8_dot(left, right)

    def test_reject_non_int8_operands(self):
        for operand, exception in ((128, ValueError), (-129, ValueError),
                                   (1.0, TypeError), (True, TypeError)):
            with self.subTest(operand=operand), self.assertRaises(exception):
                int8_dot([operand], [1])

    def test_accumulator_overflow_and_underflow(self):
        with self.assertRaises(OverflowError):
            checked_mac(2**31 - 1, 1, 1)
        with self.assertRaises(OverflowError):
            checked_mac(-(2**31), -1, 1)
        # Python could hold this value; the implemented INT32 arithmetic cannot.
        with self.assertRaises(OverflowError):
            int8_dot([-128] * 131072, [-128] * 131072)

    def test_initial_accumulator_type_and_limits(self):
        self.assertEqual(checked_mac(2**31 - 1, 0, 127), 2**31 - 1)
        self.assertEqual(checked_mac(-(2**31), 0, -128), -(2**31))
        for accumulator in (True, 1.0):
            with self.subTest(accumulator=accumulator), self.assertRaises(TypeError):
                checked_mac(accumulator, 0, 0)
        for accumulator in (2**31, -(2**31) - 1):
            with self.subTest(accumulator=accumulator), self.assertRaises(OverflowError):
                checked_mac(accumulator, 0, 0)

    def test_fp32_rounding_differs_from_python_float_reference(self):
        self.assertEqual(fp32(16777217.0), 16777216.0)
        total = fp32_mac(0, 16777216, 1)
        total = fp32_mac(total, 1, 1)
        self.assertEqual(fp32_mac(total, -16777216, 1), 0.0)
        # Wider Python floats retain the +1 that FP32 loses at 2**24.
        self.assertEqual(float_matmul([[16777216, 1, -16777216]], [[1], [1], [1]]), [[1.0]])

    def test_symmetric_quantization_and_clipping(self):
        result = quantize_int8([[-200, -127, 0, 127, 200]], scale=1)
        self.assertEqual(result.values, [[-127, -127, 0, 127, 127]])
        self.assertEqual(result.clipped_values, 2)

    def test_rounding_ties_to_even(self):
        result = quantize_int8([[-0.75, -0.25, 0.25, 0.75]], scale=0.5)
        self.assertEqual(result.values, [[-2, 0, 0, 2]])

    def test_zero_matrix_and_automatic_scale(self):
        zeros = quantize_int8([[0, 0]])
        self.assertEqual(zeros.values, [[0, 0]])
        self.assertEqual(zeros.scale, 1)
        result = quantize_int8([[-2, 0, 2]])
        self.assertEqual(result.values, [[-127, 0, 127]])
        self.assertEqual(result.scale, 2 / 127)

    def test_invalid_scales_and_nonfinite_values(self):
        for scale in (0, -1, math.inf, math.nan):
            with self.subTest(scale=scale), self.assertRaises(ValueError):
                quantize_int8([[1]], scale)
        for value in (math.inf, -math.inf, math.nan):
            with self.assertRaises(ValueError):
                quantize_int8([[value]])

    def test_product_uses_both_scales(self):
        result = int8_matmul([[2, 3]], [[4], [5]])
        self.assertEqual(result, [[23]])
        reconstructed = dequantize_accumulators(result, 0.5, 0.25)
        self.assertEqual(reconstructed, [[2.875]])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ArithmeticTests)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
if not result.wasSuccessful():
    raise AssertionError("An arithmetic check failed; review the output above.")
print(f"All {result.testsRun} arithmetic checks passed.")


test_accumulator_overflow_and_underflow (__main__.ArithmeticTests.test_accumulator_overflow_and_underflow) ... ok
test_fp32_rounding_differs_from_python_float_reference (__main__.ArithmeticTests.test_fp32_rounding_differs_from_python_float_reference) ... ok
test_hand_calculated_dot_product (__main__.ArithmeticTests.test_hand_calculated_dot_product) ... ok
test_hand_calculated_matrix (__main__.ArithmeticTests.test_hand_calculated_matrix) ... ok
test_initial_accumulator_type_and_limits (__main__.ArithmeticTests.test_initial_accumulator_type_and_limits) ... ok
test_invalid_scales_and_nonfinite_values (__main__.ArithmeticTests.test_invalid_scales_and_nonfinite_values) ... ok
test_product_uses_both_scales (__main__.ArithmeticTests.test_product_uses_both_scales) ... ok
test_reject_invalid_shapes (__main__.ArithmeticTests.test_reject_invalid_shapes) ... ok
test_reject_non_int8_operands (__main__.ArithmeticTests.test_reject_non_int8_operands) ... ok
test_rounding_ties_to_even (__main__.Arithme

## What this establishes

The arithmetic rules work on small, independently checked examples. The next
project step models **how many cycles those calculations and memory transfers
need**. Digit-recognition accuracy and accelerator speed come later.
